# Assignment 1 - Building a Vision Model with Keras

In this assignment, you will build a simple vision model using Keras. The goal is to classify images from the Fashion MNIST dataset, which contains images of clothing items.

You will:
1. Load and inspect the Fashion MNIST dataset.
2. Run a simple baseline model to establish a performance benchmark.
3. Build and evaluate a simple CNN model, choosing appropriate loss and metrics.
4. Design and run controlled experiments on one hyperparameter (e.g., number of filters, kernel size, etc.) and one regularization technique (e.g., dropout, L2 regularization).
5. Analyze the results and visualize the model's performance.

# 1. Loading and Inspecting the Dataset

Fashion MNIST is a dataset of grayscale images of clothing items, with 10 classes. Each image is 28x28 pixels, like the MNIST dataset of handwritten digits. Keras provides a convenient way to load this dataset.

In this section, you should:

- [ ] Inspect the shapes of the training and test sets to confirm their size and structure.
- [ ] Convert the labels to one-hot encoded format if necessary. (There is a utility function in Keras for this.)
- [ ] Visualize a few images from the dataset to understand what the data looks like.

In [ ]:
from tensorflow.keras.datasets import fashion_mnist
(X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()

# Normalize the pixel values to be between 0 and 1
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

# Classes in the Fashion MNIST dataset
class_names = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat", "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
# Inspect the shapes of the datasets

print("=== Dataset Shapes ===")
print("Training data shape:", X_train.shape)
print("Training labels shape:", y_train.shape)
print("Test data shape:", X_test.shape)
print("Test labels shape:", y_test.shape)
# Convert labels to one-hot encoding
from tensorflow.keras.utils import to_categorical
y_train_one_hot = to_categorical(y_train, num_classes=10)
y_test_one_hot = to_categorical(y_test, num_classes=10)

print(to_categorical)



=== Dataset Shapes ===
Training data shape: (60000, 28, 28)
Training labels shape: (60000,)
Test data shape: (10000, 28, 28)
Test labels shape: (10000,)
<function to_categorical at 0x7db3c416be20>


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
# Verify the data looks as expected
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)
print("\nData range - Min: {:.3f}, Max: {:.3f}".format(X_train.min(), X_train.max()))
print("Data type:", X_train.dtype)

# plt.figure(figsize=(15, 10))

samples_per_class = []
for class_id in range(10):
    # Find the first occurrence of each class
    idx = np.where(y_train == class_id)[0][0]
    samples_per_class.append((X_train[idx], y_train[idx], class_names[class_id]))

# Plot samples from each class
# for i, (image, label, class_name) in enumerate(samples_per_class):
#     plt.subplot(2, 5, i+1)
#     plt.imshow(image, cmap='gray')
#     plt.title(f'Class {label}: {class_name}')
#     plt.axis('off')
# plt.tight_layout()
# plt.show()

X_train shape: (60000, 28, 28)
y_train shape: (60000,)
X_test shape: (10000, 28, 28)
y_test shape: (10000,)

Data range - Min: 0.000, Max: 1.000
Data type: float32


Reflection: Does the data look as expected? How is the quality of the images? Are there any issues with the dataset that you notice?

**Your answer here**

# 2. Baseline Model

In this section, you will create a linear regression model as a baseline. This model will not use any convolutional layers, but it will help you understand the performance of a simple model on this dataset.
You should:
- [ ] Create a simple linear regression model using Keras.
- [ ] Compile the model with an appropriate loss function and optimizer.
- [ ] Train the model on the training set and evaluate it on the test set.

A linear regression model can be created using the `Sequential` API in Keras. Using a single `Dense` layer with no activation function is equivalent to a simple linear regression model. Make sure that the number of units in the output layer matches the number of classes in the dataset.

Note that for this step, we will need to use `Flatten` to convert the 2D images into 1D vectors before passing them to the model. Put a `Flatten()` layer as the first layer in your model so that the 2D image data can be flattened into 1D vectors.

In [ ]:
from keras.models import Sequential
from keras.layers import Dense, Flatten

# Create a simple linear regression model
model = Sequential()
# You can use `model.add(<layer>)` to add layers to the model
model.add(Flatten())
model.add(Dense(10))

# Compile the model using `model.compile()`
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Train the model with `model.fit()`
history = model.fit(
    X_train,
    y_train_one_hot,
    batch_size=32,
    epochs=10,
    validation_data=(X_test, y_test_one_hot),
    verbose=1
)

# Evaluate the model with `model.evaluate()`
test_loss, test_acc = model.evaluate(X_test, y_test_one_hot, verbose=0)
print(f"Test loss: {test_loss:.4f} | Test acc: {test_acc:.4f}")

Epoch 1/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.1950 - loss: 9.9420 - val_accuracy: 0.2115 - val_loss: 10.0241
Epoch 2/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.2200 - loss: 10.5508 - val_accuracy: 0.2008 - val_loss: 10.5606
Epoch 3/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.2110 - loss: 10.9549 - val_accuracy: 0.2814 - val_loss: 10.5752
Epoch 4/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.2943 - loss: 10.4535 - val_accuracy: 0.2702 - val_loss: 11.4762
Epoch 5/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.2809 - loss: 10.7292 - val_accuracy: 0.1876 - val_loss: 12.4786
Epoch 6/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.1953 - loss: 12.5695 - val_accuracy: 0.1876 - val_loss: 12.4803
Epoch 7/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.1935 - loss: 12.6569 - val_accuracy: 0.1876 - val_loss: 12.4803
Epoch 8/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.1953 - lo

Reflection: What is the performance of the baseline model? How does it compare to what you expected? Why do you think the performance is at this level?

**Your answer here**

Accuracy is 18% which is not great. it is lower than expected and not much higher than guessing. the performance is at this level because in a linear model, it treats each pixel independently

# 3. Building and Evaluating a Simple CNN Model

In this section, you will build a simple Convolutional Neural Network (CNN) model using Keras. A convolutional neural network is a type of deep learning model that is particularly effective for image classification tasks. Unlike the basic neural networks we have built in the labs, CNNs can accept images as input without needing to flatten them into vectors.

You should:
- [ ] Build a simple CNN model with at least one convolutional layer (to learn spatial hierarchies in images) and one fully connected layer (to make predictions).
- [ ] Compile the model with an appropriate loss function and metrics for a multi-class classification problem.
- [ ] Train the model on the training set and evaluate it on the test set.

Convolutional layers are designed to accept inputs with three dimensions: height, width and channels (e.g., RGB for color images). For grayscale images like those in Fashion MNIST, the input shape will be (28, 28, 1).

When you progress from the convolutional layers to the fully connected layers, you will need to flatten the output of the convolutional layers. This can be done using the `Flatten` layer in Keras, which doesn't require any parameters.

In [ ]:
from keras.layers import Conv2D
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout


# Reshape the data to include the channel dimension
X_train = X_train.reshape(-1, 28, 28, 1)
X_test = X_test.reshape(-1, 28, 28, 1)

input_shape = (28, 28, 1)

# Create a simple CNN model
model = Sequential([
    Conv2D(32, (3,3), activation="relu", padding="same", input_shape=input_shape),
    MaxPooling2D((2,2)),

    Conv2D(64, (3,3), activation="relu", padding="same"),
    MaxPooling2D((2,2)),

    Flatten(),
    Dense(128, activation="relu"),
    Dropout(0.3),
    Dense(num_classes, activation=None)  # logits
])

use_sparse = True
if use_sparse:
    loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
    y_tr, y_te = y_train, y_test
else:
    loss = tf.keras.losses.CategoricalCrossentropy(from_logits=True)
    # ensure one-hot float32 if you go this route
    y_tr = y_train_one_hot.astype("float32")
    y_te = y_test_one_hot.astype("float32")

model.compile(optimizer="adam", loss=loss, metrics=["accuracy"])

# Train the model
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True, monitor="val_accuracy")
]
history = model.fit(
    X_train, y_tr,
    validation_data=(X_test, y_te),
    epochs=10,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)
# Evaluate the model
test_loss, test_acc = model.evaluate(X_test, y_te, verbose=0)
print(f"Test loss: {test_loss:.4f} | Test acc: {test_acc:.4f}")

NameError: name 'num_classes' is not defined

Reflection: Did the CNN model perform better than the baseline model? If so, by how much? What do you think contributed to this improvement?

**Your answer here**

# 3. Designing and Running Controlled Experiments

In this section, you will design and run controlled experiments to improve the model's performance. You will focus on one hyperparameter and one regularization technique.
You should:
- [ ] Choose one hyperparameter to experiment with (e.g., number of filters, kernel size, number of layers, etc.) and one regularization technique (e.g., dropout, L2 regularization). For your hyperparameter, you should choose at least three different values to test (but there is no upper limit). For your regularization technique, simply test the presence or absence of the technique.
- [ ] Run experiments by modifying the model architecture or hyperparameters, and evaluate the performance of each model on the test set.
- [ ] Record the results of your experiments, including the test accuracy and any other relevant metrics.
- [ ] Visualize the results of your experiments using plots or tables to compare the performance of different models.

The best way to run your experiments is to create a `for` loop that iterates over a range of values for the hyperparameter you are testing. For example, if you are testing different numbers of filters, you can create a loop that runs the model with 32, 64, and 128 filters. Within the loop, you can compile and train the model, then evaluate it on the test set. After each iteration, you can store the results in a list or a dictionary for later analysis.

Note: It's critical that you re-initialize the model (by creating a new instance of the model) before each experiment. If you don't, the model will retain the weights from the previous experiment, which can lead to misleading results.

In [ ]:
# A. Test Hyperparameters
from datetime import datetime
import pandas as pd
import tensorflow as tf

X_train_hp = X_train.astype("float32")
X_test_hp  = X_test.astype("float32")
input_shape = X_train_hp.shape[1:]
num_classes = 10

def build_model(num_filters: int) -> tf.keras.Model:
    model = Sequential([
        Conv2D(num_filters, (3,3), activation="relu", padding="same", input_shape=input_shape),
        MaxPooling2D((2,2)),
        Conv2D(64, (3,3), activation="relu", padding="same"),
        MaxPooling2D((2,2)),

        Flatten(),
        Dense(128, activation="relu"),
        Dropout(0.3),
        Dense(num_classes, activation=None)   # logits
    ])
    model.compile(
        optimizer="adam",
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=["accuracy"]
    )
    return model

filters_grid = [16, 32, 128]
epochs = 10
batch_size = 64

results = []
histories = {}

for nf in filters_grid:
  model = build_model(nf)

  history = model.fit(
        X_train_hp, y_train,
        validation_data=(X_test_hp, y_test),
        epochs=epochs,
        batch_size=batch_size,
        verbose=1
    )

  test_loss, test_acc = model.evaluate(X_test_hp, y_test, verbose=0)
  results.append({
        "num_filters": nf,
        "best_val_acc": float(max(history.history["val_accuracy"])),
        "final_val_acc": float(history.history["val_accuracy"][-1]),
        "test_acc": float(test_acc),
        "test_loss": float(test_loss),
        "epochs_ran": len(history.history["loss"]),
        "timestamp": datetime.now().isoformat(timespec="seconds")
    })
  histories[nf] = history

df_results = pd.DataFrame(results).sort_values("test_acc", ascending=False)
print(df_results.to_string(index=False))

Epoch 1/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 49s 51ms/step - accuracy: 0.7525 - loss: 0.6854 - val_accuracy: 0.8671 - val_loss: 0.3592
Epoch 2/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 82s 51ms/step - accuracy: 0.8758 - loss: 0.3458 - val_accuracy: 0.8934 - val_loss: 0.2948
Epoch 3/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 48s 51ms/step - accuracy: 0.8950 - loss: 0.2925 - val_accuracy: 0.8921 - val_loss: 0.2936
Epoch 4/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 46s 50ms/step - accuracy: 0.9062 - loss: 0.2575 - val_accuracy: 0.9060 - val_loss: 0.2589
Epoch 5/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 82s 49ms/step - accuracy: 0.9160 - loss: 0.2263 - val_accuracy: 0.9137 - val_loss: 0.2388
Epoch 6/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 48s 51ms/step - accuracy: 0.9248 - loss: 0.2081 - val_accuracy: 0.9144 - val_loss: 0.2332
Epoch 7/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 50s 54ms/step - accuracy: 0.9300 - loss: 0.1863 - val_accuracy: 0.9139 - val_loss: 0.2366
Epoch 8/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 50s 54ms/step - accuracy: 0.9348 - loss: 0.1719 - 

In [ ]:
# B. Test presence or absence of regularization
X_train_reg = X_train.astype("float32")
X_test_reg  = X_test.astype("float32")

def build_model(use_dropout: bool, dropout_rate: float = 0.3) -> tf.keras.Model:
    layers = [
        Conv2D(32, (3,3), activation="relu", padding="same", input_shape=input_shape),
        MaxPooling2D((2,2)),
        Conv2D(64, (3,3), activation="relu", padding="same"),
        MaxPooling2D((2,2)),
        Flatten(),
        Dense(128, activation="relu"),
    ]
    if use_dropout:
        layers.append(Dropout(dropout_rate))
    layers.append(Dense(num_classes, activation=None))

    model = Sequential(layers)
    model.compile(
        optimizer="adam",
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=["accuracy"]
    )
    return model

reg_conditions = [
    {"name": "no_dropout", "use_dropout": False},
    {"name": "with_dropout", "use_dropout": True},
]

results_reg = []
histories_reg = {}

for cond in reg_conditions:
    print(f"\n=== Training condition: {cond['name']} ===")
    model = build_model(use_dropout=cond["use_dropout"])  # re-init each time

    hist = model.fit(
        X_train_reg, y_train,
        validation_data=(X_test_reg, y_test),
        epochs=epochs,
        batch_size=batch_size,
        verbose=1
    )

    test_loss, test_acc = model.evaluate(X_test_reg, y_test, verbose=0)
    results_reg.append({
        "condition": cond["name"],
        "use_dropout": cond["use_dropout"],
        "best_val_acc": float(max(hist.history["val_accuracy"])),
        "final_val_acc": float(hist.history["val_accuracy"][-1]),
        "test_acc": float(test_acc),
        "test_loss": float(test_loss),
        "epochs_ran": len(hist.history["loss"]),
        "timestamp": datetime.now().isoformat(timespec="seconds"),
    })
    histories_reg[cond["name"]] = hist

df_reg = pd.DataFrame(results_reg).sort_values("test_acc", ascending=False)
print(df_reg.to_string(index=False))

Reflection: Report on the performance of the models you tested. Did any of the changes you made improve the model's performance? If so, which ones? What do you think contributed to these improvements? Finally, what combination of hyperparameters and regularization techniques yielded the best performance?

**Your answer here**

increasing the layers generally increased performance to a point. ie higher number of filters 128

# 5. Training Final Model and Evaluation

In this section, you will train the final model using the best hyperparameters and regularization techniques you found in the previous section. You should:
- [ ] Compile the final model with the best hyperparameters and regularization techniques.
- [ ] Train the final model on the training set and evaluate it on the test set.
- [ ] Report the final model's performance on the test set, including accuracy and any other relevant metrics.

In [ ]:
X_train_final = X_train.astype("float32")
X_test_final  = X_test.astype("float32")

def build_final_model() -> tf.keras.Model:
    model = Sequential([
        Input(shape=input_shape),
        Conv2D(128, (3,3), activation="relu", padding="same"),
        MaxPooling2D((2,2)),
        Conv2D(64, (3,3), activation="relu", padding="same"),
        MaxPooling2D((2,2)),
        Flatten(),
        Dense(128, activation="relu"),
        Dropout(0.3),
        Dense(num_classes, activation=None)
    ])
    model.compile(
        optimizer="adam",
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=[
            "accuracy",
            tf.keras.metrics.SparseTopKCategoricalAccuracy(k=3, name="top3_acc"),
        ],
    )
    return model

model = build_final_model()
model.summary()

history = model.fit(
    X_train_final, y_train,
    validation_data=(X_test_final, y_test),
    epochs=15,
    batch_size=64,
    verbose=1,
)

test_metrics = model.evaluate(X_test_final, y_test, verbose=0)
metric_names = model.metrics_names
final_report = {k: float(v) for k, v in zip(metric_names, test_metrics)}

print("=== Final Model Test Performance ===")
for k, v in final_report.items():
    if k in {"accuracy", "top3_acc"}:
        print(f"{k:>12}: {v:.4f}")
    else:
        print(f"{k:>12}: {v:.4f}")

Reflection: How does the final model's performance compare to the baseline and the CNN model? What do you think contributed to the final model's performance? If you had time, what other experiments would you run to further improve the model's performance?

**Your answer here**

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.
### Submission Parameters:
* Submission Due Date: `23:59 PM - 26/10/2025`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb)
    * The Lab 1 notebook (labs/lab_1.ipynb)
    * The Lab 2 notebook (labs/lab_2.ipynb)
    * The Lab 3 notebook (labs/lab_3.ipynb)
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/deep_learning/pull/<pr_id>`
* Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.
Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.
If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-7-help-ml`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.